<a href="https://colab.research.google.com/github/dipika-desaboyina/whisper-telugu-benchmarking/blob/main/benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import unicodedata

s1 = "é"
s2 = "e" + "\u0301"

print(s1 == s2)
print(unicodedata.normalize("NFC", s1) ==
      unicodedata.normalize("NFC", s2))
print(unicodedata.normalize("NFC", s1))

False
True
é


In [2]:
import re

text = "test text with .,;:'[]()"
text  = re.sub('[.,;:"\'()\[\]|]',"",text)

print(text)

test text with 


<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_398/2109574348.py:4: SyntaxWarning: invalid escape sequence '\['
  text  = re.sub('[.,;:"\'()\[\]|]',"",text)


In [3]:
!pip install -q faster-whisper jiwer datasets soundfile librosa


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.2 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import gc, json, unicodedata, csv, re, time, os
import numpy as np
import soundfile as sf
from pathlib import Path
from datasets import Audio, load_dataset
from faster_whisper import WhisperModel
from jiwer import wer, cer
from IPython.display import Audio as IPyAudio, display

try:
    import torch
    gpu_available = torch.cuda.is_available()
except ImportError:
    gpu_available = False

if gpu_available:
    device = "cuda"
    compute_type = "float16"
    print("GPU detected; running on cuda with float16")
else:
    device = "cpu"
    compute_type = "int8"
    print("No GPU detected; running on cpu with int8 (Runtime > Change runtime type > T4 GPU to enable one)")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("")
except Exception:
    HF_TOKEN = os.environ.get("")

if HF_TOKEN is None:
    print("WARNING: HF_TOKEN not found in Colab Secrets or environment. Gated datasets will fail to load without it.")

enabled_datasets = ["fleurs", "open_slr", "indic_voices"]

# audio clips for manual QA checks
n_audio_samples_to_check = 5

# defining config variables

language = "te" # ISO code for Telugu in Whisper
target_sampling_rate = 16000 # 16KHz
model_sizes = ["base"]
sample_size = 300
beam_size = 5

output_directory = Path("/content/drive/MyDrive/telugu-whisper-benchmark/results")
output_directory.mkdir(parents=True, exist_ok=True)
results_csv = output_directory/"wer_cer_by_model.csv"
predictions_directory = output_directory/"predictions"
predictions_directory.mkdir(exist_ok=True)
audio_samples_directory = output_directory/"audio_samples"
audio_samples_directory.mkdir(exist_ok=True)


# defining dataset language config

open_slr_config = "SLR66"
fleurs_config = "te_in"
indic_voices_config = "telugu"
indic_voices_r_config = "Telugu"
kathbath_config = "telugu"

# core components

# text normalisation

def normalise_telugu(text: str)->str:
    """applying NFC for unicode normalisation, regex based stripping for general purpose text normalisation"""
    text = unicodedata.normalize("NFC",text)
    text = text.strip()
    text = re.sub(r'[.,:;"\'\[\]()|]',"",text)
    text = re.sub(r'\s+'," ",text)
    return text.strip()

# dataset loading

def _resampling(dataset):
    return dataset.cast_column("audio", Audio(sampling_rate=target_sampling_rate))

def load_open_slr(sample_size:int):
    # public dataset — token not required, but harmless to pass anyway
    data = load_dataset("openslr/openslr", open_slr_config, split="train", token=HF_TOKEN)
    data = _resampling(data)
    data = data.select(range(min(sample_size, len(data))))
    return [(row["audio"]["array"].astype(np.float32),row["sentence"]) for row in data] # type: ignore

def load_fleurs(sample_size:int):
    data = load_dataset("google/fleurs", fleurs_config, split="train", token=HF_TOKEN)
    data = _resampling(data)
    data = data.select(range(min(sample_size, len(data))))
    return [(row["audio"]["array"].astype(np.float32),row["transcription"]) for row in data] # type: ignore

def load_indic_voices(sample_size:int):
    # genuinely requires a token per AI4Bharat's own docs
    data = load_dataset("ai4bharat/IndicVoices",indic_voices_config,split="valid", token=HF_TOKEN)
    data = _resampling(data)
    data = data.select(range(min(sample_size, len(data))))
    return [(row["audio"]["array"].astype(np.float32),row["text"]) for row in data] # type: ignore

def load_indic_voices_r(sample_size:int):
    # gated — remember, token + logged-in "Agree" click on the HF page, both needed
    data = load_dataset("ai4bharat/indicvoices_r", indic_voices_r_config, split="test", token=HF_TOKEN)
    data = _resampling(data)
    data = data.select(range(min(sample_size, len(data))))
    text_field = "text" if "text" in data.column_names else "normalized_text"
    return [(row["audio"]["array"].astype(np.float32),row[text_field]) for row in data] # type: ignore

def load_kathbath(sample_size:int):
    # gated, config/split names below unconfirmed — check the dataset page
    data = load_dataset("ai4bharat/Kathbath", kathbath_config, split="test", token=HF_TOKEN)
    data = _resampling(data)
    data = data.select(range(min(sample_size, len(data))))
    text_field = "text" if "text" in data.column_names else "sentence"
    return [(row["audio"]["array"].astype(np.float32),row[text_field]) for row in data] # type: ignore

dataset_loader = {
    "fleurs": load_fleurs,
    "open_slr": load_open_slr,
    "indic_voices": load_indic_voices,
    "indic_voices_r": load_indic_voices_r,
    "kathbath": load_kathbath,
}

def save_and_play_audio_samples(dataset_name: str, samples, n: int = n_audio_samples_to_check):
    dataset_dir = audio_samples_directory/dataset_name
    dataset_dir.mkdir(exist_ok=True)
    manifest = []
    for i, (audio_array, ref) in enumerate(samples[:n]):
        file_path = dataset_dir/f"sample_{i:02d}.wav"
        sf.write(file_path, audio_array, target_sampling_rate)
        manifest.append({
            "file": str(file_path),
            "reference": ref,
            "reference_normalised": normalise_telugu(ref),
        })
    manifest_path = dataset_dir/"manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"    saved {len(manifest)} audio samples -> {dataset_dir}/")

        for entry in manifest[:2]:
        print(f"    reference: {entry['reference']}")
        display(IPyAudio(entry["file"]))

# checkpointing

def rows_done(model_size:str, dataset_name:str)->bool:
    if not results_csv.exists():
        return False
    with open(results_csv,"r",newline="",encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["model"]==model_size and row["dataset"]==dataset_name:
                return True
    return False

def append_results(row:dict):
    write_header = not results_csv.exists()
    with open(results_csv,"a",newline="",encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)

# benchmarking

def transcribe(model: WhisperModel, samples, beam_size:int=5):
    predictions = []
    time_per_utterance = []
    for audio_array, _ in samples:
        start = time.time()
        segments,_ = model.transcribe(audio_array,language=language,beam_size=beam_size)
        text = " ".join(seg.text for seg in segments)
        time_per_utterance.append(time.time()-start)
        predictions.append(text)
    return predictions,time_per_utterance


max_utterance_seconds = 30 # guard against occasional very long clips (esp. indic_voices) blowing up RAM

def benchmark(model_sizes, sample_size: int, beam_size: int ):
    for dataset_name in enabled_datasets:
        loader = dataset_loader[dataset_name]
        print(f"\n### Dataset: {dataset_name} ###")
        try:
            samples = loader(sample_size) # type: ignore
        except Exception as e:
            print(f"    failed to load {dataset_name}: {e}  ")
            print(f"    skipping {dataset_name} for this run, fix config/token/access-request and rerun  ")
            continue

        before = len(samples)
        samples = [
            (audio, ref) for audio, ref in samples
            if len(audio) / target_sampling_rate <= max_utterance_seconds
        ]
        dropped = before - len(samples)
        if dropped:
            print(f"    dropped {dropped} clips longer than {max_utterance_seconds}s")
        print(f"    {len(samples)} samples ready   ")

        save_and_play_audio_samples(dataset_name, samples)

        for model_size in model_sizes:
            if rows_done(model_size, dataset_name):
                print(f"    {dataset_name} already benchmarked for {model_size}     ")
                continue

            print(f"    loading model: {model_size}")
            model = WhisperModel(model_size, device=device, compute_type=compute_type)

            print(f"    transcribing {dataset_name} for {len(samples)} utterances    ")
            predictions, time_per_utterance = transcribe(model,samples,beam_size=beam_size)
            references = [ref for _, ref in samples]
            norm_refs = [normalise_telugu(r) for r in references]
            norm_preds = [normalise_telugu(p) for p in predictions]

            w = wer(norm_refs, norm_preds)
            c = cer(norm_refs, norm_preds)
            total_time = sum(time_per_utterance)

            append_results({
                "model" : model_size,
                "dataset" : dataset_name,
                "sample_size" : len(samples),
                "wer" : round(w,3),
                "cer" : round(c,3),
                "total_transcription_time" : total_time,
                "avg_time_per_utterance_in_seconds" : round(total_time/len(samples),3) if samples else 0,
            })

            prediction_path = predictions_directory/f"{model_size}__{dataset_name}.jsonl"
            with open(prediction_path, "w", encoding="utf-8") as f:
                for (audio_array, ref), pred in zip(samples, predictions):
                    f.write(json.dumps({
                        "reference": ref,
                        "prediction": pred,
                        "reference_normalised" : normalise_telugu(ref),
                        "prediction_normalised" : normalise_telugu(pred),
                    }, ensure_ascii=False)+"\n")
            print(f"    WER={w:.3f}     CER:{c:.3f}     total_time={total_time:.1f} seconds     ")

            del model
            gc.collect()

        del samples
        gc.collect()

    print(f"\nDone. Results in {results_csv}, predictions in {predictions_directory}/, audio samples in {audio_samples_directory}/   ")

benchmark(model_sizes, sample_size, beam_size)

GPU detected — running on cuda with float16
Gated datasets (indic_voices, indic_voices_r, kathbath) will fail to load without it.

### Dataset: fleurs ###
    300 samples ready   
    saved 5 audio samples -> /content/drive/MyDrive/telugu-whisper-benchmark/results/audio_samples/fleurs/
    reference: డెంగ్ జియావోపింగ్ నాయకత్వంలో మొదటి ఆర్థిక సంస్కరణలు జరిగాయి


    reference: తుఫాను తీరానికి దూరంగా ఉన్నందున యునైటెడ్ స్టేట్స్ లేదా కరేబియన్కు సంభావ్య ప్రభావాన్ని అంచనా వేయడం కష్టం


    loading model: base
    transcribing fleurs for 300 utterances    
    WER=1.229     CER:1.019     total_time=418.2 seconds     

### Dataset: open_slr ###
    failed to load open_slr: Dataset scripts are no longer supported, but found openslr.py  
    skipping open_slr for this run, fix config/token/access-request and rerun  

### Dataset: indic_voices ###
    failed to load indic_voices: Dataset 'ai4bharat/IndicVoices' is a gated dataset on the Hub. You must be authenticated to access it.  
    skipping indic_voices for this run, fix config/token/access-request and rerun  

Done. Results in /content/drive/MyDrive/telugu-whisper-benchmark/results/wer_cer_by_model.csv, predictions in /content/drive/MyDrive/telugu-whisper-benchmark/results/predictions/, audio samples in /content/drive/MyDrive/telugu-whisper-benchmark/results/audio_samples/   
